# Steam Price Evaluation

This notebook locks the final project story for Steam price modeling. The task is to estimate a market-aligned paid-game list price from Steam metadata, then convert the prediction into business-facing price-alignment labels.

There are two intentionally separate use cases:

- Developer-facing launch or benchmark pricing uses only pre-release metadata, so it can support pricing decisions before reviews and player activity exist.
- Player-facing value assessment can use post-release signals such as reviews, recommendations, CCU, playtime, and Metacritic because players can observe those signals before buying.

The main model target is `log_list_price`, derived from the reconstructed `estimated_list_price`. The log target is used because Steam prices are skewed and multiplicative errors are more meaningful than treating a `$2` error on a `$3` game the same way as a `$2` error on a `$60` game.


In [ ]:
import ast
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, median_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import Pipeline
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler

pd.set_option("display.max_columns", 140)
pd.set_option("display.max_colwidth", 140)

In [ ]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "games_price_model.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
MODEL_DATA_PATH = DATA_DIR / "games_price_model.csv"

## 1. Load modeling data

The feature-engineering notebook already filters to valid paid games with estimated list prices between `$0.49` and `$99.99`. Free-to-play titles are excluded because free-game monetization is a different problem, `Discount == 100` rows are excluded because the list price cannot be reconstructed, and very high prices are capped out of scope to avoid collector packs and extreme catalog artifacts dominating the model.


In [ ]:
def parse_list(value):
    if pd.isna(value):
        return []
    return ast.literal_eval(value)


df = pd.read_csv(MODEL_DATA_PATH)
for col in ["genre_list", "tag_list", "category_list"]:
    df[col] = df[col].apply(parse_list)

print(f"Shape: {df.shape}")
print("Valid price targets:", df["valid_price_target"].sum())
print("Estimated list price range:", df["estimated_list_price"].min(), df["estimated_list_price"].max())
df.head()

In [ ]:
df["estimated_list_price"].describe(percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).to_frame("estimated_list_price")

## 2. Feature sets

This first pass uses pre-release features only. Outcome variables such as reviews, recommendations, CCU, and playtime are held back for a later gamer-facing value model.

In [ ]:
TARGET = "log_list_price"

ID_COLS = ["AppID", "Name"]
NUMERIC_FEATURES = [
    "required_age_clipped",
    "platform_count",
    "Release year",
    "release_age_years",
    "dlc_count_clipped",
    "achievements_clipped",
    "log1p_dlc_count",
    "log1p_achievements",
    "genre_count",
    "tag_count",
    "category_count",
]
BOOLEAN_FEATURES = ["Windows", "Mac", "Linux"]
MULTILABEL_FEATURES = ["genre_list", "tag_list", "category_list"]

TAG_MIN_COUNT = 100
CATEGORY_MIN_COUNT = 100


def label_counts(series):
    return series.explode().loc[lambda s: s.notna() & s.ne("")].value_counts()


genre_vocab = sorted(label_counts(df["genre_list"]).index.tolist())
tag_vocab = sorted(label_counts(df["tag_list"]).loc[lambda s: s >= TAG_MIN_COUNT].index.tolist())
category_vocab = sorted(label_counts(df["category_list"]).loc[lambda s: s >= CATEGORY_MIN_COUNT].index.tolist())

feature_summary = pd.Series({
    "rows": len(df),
    "numeric_features": len(NUMERIC_FEATURES),
    "boolean_features": len(BOOLEAN_FEATURES),
    "genre_features": len(genre_vocab),
    "tag_features": len(tag_vocab),
    "category_features": len(category_vocab),
})
feature_summary.to_frame("count")

## 3. Build sparse feature matrix

Ridge and KNN both use the same pre-release feature matrix. Numeric features are imputed and standardized; multi-label features are multi-hot encoded.

In [ ]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

numeric_transformer = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, NUMERIC_FEATURES),
        ("boolean", SimpleImputer(strategy="most_frequent"), BOOLEAN_FEATURES),
    ],
    remainder="drop",
    sparse_threshold=1.0,
)

genre_binarizer = MultiLabelBinarizer(classes=genre_vocab, sparse_output=True)
tag_binarizer = MultiLabelBinarizer(classes=tag_vocab, sparse_output=True)
category_binarizer = MultiLabelBinarizer(classes=category_vocab, sparse_output=True)


def keep_vocab(items, vocab):
    vocab = set(vocab)
    return [item for item in items if item in vocab]


def build_feature_matrix(frame, fit=False):
    frame_for_transform = frame.copy()
    frame_for_transform[BOOLEAN_FEATURES] = frame_for_transform[BOOLEAN_FEATURES].astype(float)
    frame_for_transform["genre_list"] = frame_for_transform["genre_list"].apply(lambda items: keep_vocab(items, genre_vocab))
    frame_for_transform["tag_list"] = frame_for_transform["tag_list"].apply(lambda items: keep_vocab(items, tag_vocab))
    frame_for_transform["category_list"] = frame_for_transform["category_list"].apply(lambda items: keep_vocab(items, category_vocab))

    if fit:
        numeric_matrix = numeric_transformer.fit_transform(frame_for_transform)
        genre_matrix = genre_binarizer.fit_transform(frame_for_transform["genre_list"])
        tag_matrix = tag_binarizer.fit_transform(frame_for_transform["tag_list"])
        category_matrix = category_binarizer.fit_transform(frame_for_transform["category_list"])
    else:
        numeric_matrix = numeric_transformer.transform(frame_for_transform)
        genre_matrix = genre_binarizer.transform(frame_for_transform["genre_list"])
        tag_matrix = tag_binarizer.transform(frame_for_transform["tag_list"])
        category_matrix = category_binarizer.transform(frame_for_transform["category_list"])

    numeric_matrix = sparse.csr_matrix(numeric_matrix)
    return sparse.hstack([numeric_matrix, genre_matrix, tag_matrix, category_matrix], format="csr")


print("Feature matrix builder ready")



## 4. Evaluation helpers

Models train on log price, but evaluation is reported in dollars for business readability. The primary metric is dollar MAE because pricing errors are naturally interpreted as expected dollar miss. Median absolute error is included because price data has outliers, RMSE is included to expose large mistakes, and the within-25% / within-50% rates connect regression quality to the final price-alignment product behavior.


In [ ]:
def to_price(log_values):
    return np.expm1(log_values).clip(min=0.49)


def evaluate_predictions(model_name, y_true_log, y_pred_log):
    actual = to_price(y_true_log)
    predicted = to_price(y_pred_log)
    errors = np.abs(actual - predicted)
    rmse = mean_squared_error(actual, predicted) ** 0.5

    return {
        "model": model_name,
        "mae_dollars": mean_absolute_error(actual, predicted),
        "median_ae_dollars": median_absolute_error(actual, predicted),
        "rmse_dollars": rmse,
        "mean_actual_price": actual.mean(),
        "mean_predicted_price": predicted.mean(),
        "pct_within_25pct": (np.maximum(actual, predicted) / np.minimum(actual, predicted) <= 1.25).mean() * 100,
        "pct_within_50pct": (np.maximum(actual, predicted) / np.minimum(actual, predicted) <= 1.50).mean() * 100,
    }


def price_band_table(frame, y_pred_log):
    result = frame[["estimated_list_price"]].copy()
    result["predicted_price"] = to_price(y_pred_log)
    result["abs_error"] = (result["estimated_list_price"] - result["predicted_price"]).abs()
    result["price_band"] = pd.cut(
        result["estimated_list_price"],
        bins=[0, 2.99, 4.99, 9.99, 19.99, 49.99, 100],
        labels=["0.49-2.99", "3-4.99", "5-9.99", "10-19.99", "20-49.99", "50-99.99"],
        include_lowest=True,
    )
    return (
        result.groupby("price_band", observed=False)
        .agg(
            games=("estimated_list_price", "size"),
            median_actual=("estimated_list_price", "median"),
            median_predicted=("predicted_price", "median"),
            mae=("abs_error", "mean"),
            median_ae=("abs_error", "median"),
        )
    )

## 5. Market-scope sensitivity

These scopes test whether removing near-zero-traction games improves price modeling. The filters are based on observable market evidence, not subjective title quality.


In [ ]:
scope_masks = {
    "full_valid": pd.Series(True, index=df.index),
    "review_count >= 10": df["review_count"] >= 10,
    "review_count >= 50": df["review_count"] >= 50,
    "recommendations > 0": df["log1p_recommendations"] > 0,
}

scope_rows = []
for scope_name, mask in scope_masks.items():
    scope_df = df.loc[mask].copy()
    scope_rows.append({
        "scope": scope_name,
        "rows": len(scope_df),
        "pct_of_full": len(scope_df) / len(df) * 100,
        "median_price": scope_df["estimated_list_price"].median(),
        "mean_price": scope_df["estimated_list_price"].mean(),
        "median_review_count": scope_df["review_count"].median(),
        "median_recommendations_log": scope_df["log1p_recommendations"].median(),
        "reconstructed_pct": scope_df["price_target_source"].eq("reconstructed").mean() * 100,
    })

scope_summary_df = pd.DataFrame(scope_rows)
scope_summary_df.round(2)


In [ ]:
def train_models_for_scope(scope_df, alpha=10.0, random_state=42):
    train_scope_df, test_scope_df = train_test_split(scope_df, test_size=0.20, random_state=random_state)
    y_scope_train = train_scope_df[TARGET].to_numpy()
    y_scope_test = test_scope_df[TARGET].to_numpy()

    X_scope_train = build_feature_matrix(train_scope_df, fit=True)
    X_scope_test = build_feature_matrix(test_scope_df, fit=False)

    median_model = DummyRegressor(strategy="median")
    median_model.fit(X_scope_train, y_scope_train)
    median_pred_log = median_model.predict(X_scope_test)

    ridge_model = Ridge(alpha=alpha, random_state=random_state)
    ridge_model.fit(X_scope_train, y_scope_train)
    ridge_pred_log = ridge_model.predict(X_scope_test)

    median_metrics = evaluate_predictions("Median baseline", y_scope_test, median_pred_log)
    ridge_metrics = evaluate_predictions("Ridge", y_scope_test, ridge_pred_log)

    shared_metrics = {
        "train_rows": len(train_scope_df),
        "test_rows": len(test_scope_df),
        "test_median_price": test_scope_df["estimated_list_price"].median(),
        "test_mean_price": test_scope_df["estimated_list_price"].mean(),
    }
    median_metrics.update(shared_metrics)
    ridge_metrics.update(shared_metrics)

    return {
        "median_metrics": median_metrics,
        "ridge_metrics": ridge_metrics,
        "median_model": median_model,
        "ridge_model": ridge_model,
        "train_df": train_scope_df,
        "test_df": test_scope_df,
        "X_train": X_scope_train,
        "X_test": X_scope_test,
        "median_pred_log": median_pred_log,
        "ridge_pred_log": ridge_pred_log,
    }


In [ ]:
scope_models = {}
scope_metrics = []

for scope_name, mask in scope_masks.items():
    scope_df = df.loc[mask].copy()
    scope_result = train_models_for_scope(scope_df)
    scope_models[scope_name] = scope_result

    for metrics in [scope_result["median_metrics"], scope_result["ridge_metrics"]]:
        row = metrics.copy()
        row["scope"] = scope_name
        scope_metrics.append(row)

scope_results = pd.DataFrame(scope_metrics)
scope_results = scope_results[[
    "scope", "model", "train_rows", "test_rows", "mae_dollars", "median_ae_dollars", "rmse_dollars",
    "pct_within_25pct", "pct_within_50pct", "test_median_price", "test_mean_price"
]]

baseline_mae = (
    scope_results.loc[scope_results["model"].eq("Median baseline"), ["scope", "mae_dollars"]]
    .rename(columns={"mae_dollars": "scope_median_baseline_mae"})
)
scope_results = scope_results.merge(baseline_mae, on="scope", how="left")
scope_results["mae_improvement_vs_scope_median"] = (
    scope_results["scope_median_baseline_mae"] - scope_results["mae_dollars"]
)
scope_results["mae_improvement_pct_vs_scope_median"] = (
    scope_results["mae_improvement_vs_scope_median"] / scope_results["scope_median_baseline_mae"] * 100
)

scope_results.sort_values(["scope", "model"]).round(3)


## 6. KNN on selected market scope

If a filtered scope improves Ridge performance, run the KNN median-price baseline on that scope as a comparable-games-style estimator. The default selected scope is `review_count >= 10`, which removes completely unvalidated games while keeping enough market breadth.


In [ ]:
K_NEIGHBORS = 25
SELECTED_SCOPE = "review_count >= 10"

selected = scope_models[SELECTED_SCOPE]
selected_train_df = selected["train_df"]
selected_test_df = selected["test_df"]
X_selected_train = selected["X_train"]
X_selected_test = selected["X_test"]
y_selected_train = selected_train_df[TARGET].to_numpy()
y_selected_test = selected_test_df[TARGET].to_numpy()

selected_knn = NearestNeighbors(n_neighbors=K_NEIGHBORS, metric="cosine", algorithm="brute", n_jobs=-1)
selected_knn.fit(X_selected_train)
selected_distances, selected_indices = selected_knn.kneighbors(X_selected_test)
selected_neighbor_prices = selected_train_df.iloc[selected_indices.ravel()]["estimated_list_price"].to_numpy().reshape(selected_indices.shape)
selected_knn_pred_price = np.median(selected_neighbor_prices, axis=1)
selected_knn_pred_log = np.log1p(selected_knn_pred_price)

selected_results = [
    evaluate_predictions(f"Median baseline ({SELECTED_SCOPE})", y_selected_test, selected["median_pred_log"]),
    evaluate_predictions(f"Ridge ({SELECTED_SCOPE})", y_selected_test, selected["ridge_pred_log"]),
    evaluate_predictions(f"KNN median k={K_NEIGHBORS} ({SELECTED_SCOPE})", y_selected_test, selected_knn_pred_log),
]
selected_results_df = pd.DataFrame(selected_results)
selected_baseline_mae = selected_results_df.loc[
    selected_results_df["model"].str.startswith("Median baseline"), "mae_dollars"
].iloc[0]
selected_results_df["mae_improvement_vs_scope_median"] = selected_baseline_mae - selected_results_df["mae_dollars"]
selected_results_df["mae_improvement_pct_vs_scope_median"] = (
    selected_results_df["mae_improvement_vs_scope_median"] / selected_baseline_mae * 100
)
selected_results_df.round(3)



In [ ]:
selected_predictions = selected_test_df[[
    "AppID", "Name", "estimated_list_price", "current_price", "Discount", "price_target_source",
    "Genres", "Tags", "Categories", "review_count", "log1p_recommendations"
]].copy()
selected_predictions["predicted_market_price"] = to_price(selected_knn_pred_log)
selected_predictions["price_ratio"] = selected_predictions["estimated_list_price"] / selected_predictions["predicted_market_price"]
selected_predictions["price_alignment"] = pd.cut(
    selected_predictions["price_ratio"],
    bins=[0, 0.60, 1.50, np.inf],
    labels=["below-market", "market-aligned", "above-market"],
)

POPULAR_EXAMPLE_COLS = [
    "Name", "estimated_list_price", "predicted_market_price", "price_ratio", "price_alignment",
    "Discount", "price_target_source", "review_count", "log1p_recommendations",
]

selected_predictions.sort_values(
    ["log1p_recommendations", "review_count"], ascending=False
).loc[:, POPULAR_EXAMPLE_COLS].head(20)



## 7. KNN `k` sensitivity on selected scope

The previous KNN result used `k=25` as a reasonable starting point. This section checks whether that choice is stable on the selected `review_count >= 10` scope.


In [ ]:
def evaluate_knn_for_k(k, X_train, X_test, train_frame, y_test_log):
    model = NearestNeighbors(n_neighbors=k, metric="cosine", algorithm="brute", n_jobs=-1)
    model.fit(X_train)
    distances, indices = model.kneighbors(X_test)
    neighbor_prices = train_frame.iloc[indices.ravel()]["estimated_list_price"].to_numpy().reshape(indices.shape)
    pred_price = np.median(neighbor_prices, axis=1)
    pred_log = np.log1p(pred_price)
    metrics = evaluate_predictions(f"KNN median k={k}", y_test_log, pred_log)
    metrics["k"] = k
    return metrics, pred_log

K_VALUES = [15]
# Uncomment the line below to reexplore, results point to k=15 as a strong choice for this dataset
# K_VALUES = [3, 5, 10, 15, 25, 50, 75, 100]

knn_k_metrics = []
knn_k_predictions = {}

for k in K_VALUES:
    metrics, pred_log = evaluate_knn_for_k(
        k, X_selected_train, X_selected_test, selected_train_df, y_selected_test
    )
    knn_k_metrics.append(metrics)
    knn_k_predictions[k] = pred_log


In [ ]:
knn_k_results = pd.DataFrame(knn_k_metrics)
selected_baseline_mae = selected_results_df.loc[
    selected_results_df["model"].str.startswith("Median baseline"), "mae_dollars"
].iloc[0]
knn_k_results["mae_improvement_vs_scope_median"] = selected_baseline_mae - knn_k_results["mae_dollars"]
knn_k_results["mae_improvement_pct_vs_scope_median"] = (
    knn_k_results["mae_improvement_vs_scope_median"] / selected_baseline_mae * 100
)
knn_k_results = knn_k_results[[
    "k", "mae_dollars", "median_ae_dollars", "rmse_dollars",
    "pct_within_25pct", "pct_within_50pct",
    "mae_improvement_vs_scope_median", "mae_improvement_pct_vs_scope_median"
]].sort_values("mae_dollars")

BEST_K = int(knn_k_results.iloc[0]["k"])
best_knn_pred_log = knn_k_predictions[BEST_K]

print("Best k by MAE:", BEST_K)
knn_k_results.round(3)


## 8. Percentile-based alignment labels

This was a diagnostic experiment to inspect the residual distribution on the selected test scope. It is useful for comparison, but we do not plan to carry percentile labels into the main project story.


In [ ]:
label_predictions = selected_test_df[[
    "AppID", "Name", "estimated_list_price", "current_price", "Discount", "price_target_source",
    "Genres", "Tags", "Categories", "review_count", "log1p_recommendations"
]].copy()
label_predictions["predicted_market_price"] = to_price(best_knn_pred_log)
label_predictions["price_ratio"] = label_predictions["estimated_list_price"] / label_predictions["predicted_market_price"]
label_predictions["log_residual"] = (
    np.log1p(label_predictions["estimated_list_price"])
    - np.log1p(label_predictions["predicted_market_price"])
)

label_predictions["price_alignment_fixed"] = pd.cut(
    label_predictions["price_ratio"],
    bins=[0, 0.60, 1.50, np.inf],
    labels=["below-market", "market-aligned", "above-market"],
)

LOWER_PERCENTILE = 0.20
UPPER_PERCENTILE = 0.80
lower_cutoff = label_predictions["log_residual"].quantile(LOWER_PERCENTILE)
upper_cutoff = label_predictions["log_residual"].quantile(UPPER_PERCENTILE)

label_predictions["price_alignment_percentile"] = np.select(
    [
        label_predictions["log_residual"] < lower_cutoff,
        label_predictions["log_residual"] > upper_cutoff,
    ],
    ["below-market", "above-market"],
    default="market-aligned",
)

print("Best KNN k used for labels:", BEST_K)
print("Lower log-residual cutoff:", lower_cutoff)
print("Upper log-residual cutoff:", upper_cutoff)

label_predictions[[
    "estimated_list_price", "predicted_market_price", "price_ratio",
    "log_residual", "price_alignment_fixed", "price_alignment_percentile"
]].head()


In [ ]:
fixed_counts = label_predictions["price_alignment_fixed"].value_counts().rename("fixed_count")
percentile_counts = label_predictions["price_alignment_percentile"].value_counts().rename("percentile_count")

display(pd.concat([fixed_counts, percentile_counts], axis=1).fillna(0).astype(int))

label_crosstab = pd.crosstab(
    label_predictions["price_alignment_fixed"],
    label_predictions["price_alignment_percentile"],
    margins=True,
)
label_crosstab


In [ ]:
PERCENTILE_EXAMPLE_COLS = [
    "Name", "estimated_list_price", "predicted_market_price", "price_ratio", "log_residual",
    "price_alignment_fixed", "price_alignment_percentile", "Discount", "price_target_source",
    "Genres", "review_count", "log1p_recommendations",
]

label_predictions.sort_values(
    ["log1p_recommendations", "review_count"], ascending=False
).loc[:, PERCENTILE_EXAMPLE_COLS].head(25)


## 9. Alignment-threshold takeaways

Since the selected-scope experiments, the main picture is stable:

- `review_count >= 10` is the practical default scope.
- `0.6-1.5` is the working fixed alignment rule.
- Percentile labels are useful as a diagnostic, but not as the main business-facing output.
- KNN is stable around `k=10..25`, with only small MAE differences between the best values.

The next question was how much to widen the fixed ratio rule before it became too loose. The cells below keep the threshold set editable and let us compare a few candidate ranges on the selected scope.


In [ ]:
ALIGNMENT_THRESHOLD_PRESETS = {
    "strict_75_125": (0.75, 1.25),
    "balanced_67_150": (2 / 3, 1.50),
    "recommended_60_150": (0.60, 1.50),
}

ACTIVE_ALIGNMENT_PRESET = "recommended_60_150"
ACTIVE_ALIGNMENT_LOWER, ACTIVE_ALIGNMENT_UPPER = ALIGNMENT_THRESHOLD_PRESETS[ACTIVE_ALIGNMENT_PRESET]

alignment_check_df = selected_test_df[[
    "AppID", "Name", "estimated_list_price", "current_price", "Discount", "price_target_source",
    "Genres", "Tags", "Categories", "review_count", "log1p_recommendations"
]].copy()
alignment_check_df["predicted_market_price"] = to_price(best_knn_pred_log)
alignment_check_df["price_ratio"] = alignment_check_df["estimated_list_price"] / alignment_check_df["predicted_market_price"]


In [ ]:
def assign_price_alignment(price_ratio, lower, upper):
    return np.select(
        [price_ratio < lower, price_ratio > upper],
        ["below-market", "above-market"],
        default="market-aligned",
    )

alignment_sensitivity_rows = []
for preset_name, (lower, upper) in ALIGNMENT_THRESHOLD_PRESETS.items():
    labels = assign_price_alignment(alignment_check_df["price_ratio"], lower, upper)
    counts = pd.Series(labels).value_counts()
    total = len(labels)
    alignment_sensitivity_rows.append({
        "preset": preset_name,
        "lower": lower,
        "upper": upper,
        "below_count": int(counts.get("below-market", 0)),
        "aligned_count": int(counts.get("market-aligned", 0)),
        "above_count": int(counts.get("above-market", 0)),
        "below_pct": counts.get("below-market", 0) / total * 100,
        "aligned_pct": counts.get("market-aligned", 0) / total * 100,
        "above_pct": counts.get("above-market", 0) / total * 100,
    })

alignment_sensitivity_df = pd.DataFrame(alignment_sensitivity_rows).sort_values("aligned_pct", ascending=False)
alignment_sensitivity_df.round(2)


In [ ]:
alignment_check_df["price_alignment_active"] = assign_price_alignment(
    alignment_check_df["price_ratio"], ACTIVE_ALIGNMENT_LOWER, ACTIVE_ALIGNMENT_UPPER
)

alignment_check_df.sort_values(
    ["log1p_recommendations", "review_count"], ascending=False
).loc[:, [
    "Name", "estimated_list_price", "predicted_market_price", "price_ratio",
    "price_alignment_active", "Discount", "price_target_source", "review_count",
    "log1p_recommendations"
]].head(20)


## 10. XGBoost benchmark on selected scope

This is the direct boosting test on the same `review_count >= 10` split. The model is the improvement step after the simpler baselines: the median baseline proves the task is non-trivial, Ridge tests a fast linear model for sparse metadata, KNN tests comparable-game pricing, and XGBoost tests whether nonlinear interactions between metadata fields improve dollar MAE.

XGBoost is not treated as automatically better just because it is more complex. It needs to beat the selected-scope median baseline and the simpler pre-release baselines by enough to justify the additional complexity.


In [ ]:
try:
    import xgboost as xgb
except ImportError as exc:
    raise ImportError(
        "xgboost is not installed in the active environment. Install it from requirements.txt and rerun this cell."
    ) from exc

xgb_model = xgb.XGBRegressor(
    objective="reg:squarederror",
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=1.0,
    tree_method="hist",
    random_state=42,
    n_jobs=-1,
)
xgb_model.fit(X_selected_train, y_selected_train)
xgb_pred_log = xgb_model.predict(X_selected_test)

xgb_metrics = evaluate_predictions("XGBoost", y_selected_test, xgb_pred_log)
xgb_baseline_mae = selected_results_df.loc[
    selected_results_df["model"].eq("Median baseline (review_count >= 10)"), "mae_dollars"
].iloc[0]
xgb_metrics["mae_improvement_vs_scope_median"] = xgb_baseline_mae - xgb_metrics["mae_dollars"]
xgb_metrics["mae_improvement_pct_vs_scope_median"] = (
    xgb_metrics["mae_improvement_vs_scope_median"] / xgb_baseline_mae * 100
)

xgb_results_df = pd.DataFrame([xgb_metrics])[[
    "model", "mae_dollars", "median_ae_dollars", "rmse_dollars",
    "pct_within_25pct", "pct_within_50pct",
    "mae_improvement_vs_scope_median", "mae_improvement_pct_vs_scope_median",
]]

xgb_results_df.round(3)


## 11. XGBoost feature importance

Feature importance is used here as model interpretation, not as causal proof. The table shows which encoded metadata signals the boosted model relied on most when estimating market-aligned list price. This is useful for explaining the improvement model in plain language and for checking that the model is using plausible pricing signals rather than accidental identifiers.


In [ ]:
feature_names = (
    NUMERIC_FEATURES
    + BOOLEAN_FEATURES
    + [f"genre::{name}" for name in genre_vocab]
    + [f"tag::{name}" for name in tag_vocab]
    + [f"category::{name}" for name in category_vocab]
)

if len(feature_names) != X_selected_train.shape[1]:
    raise ValueError(
        f"Feature-name mismatch: {len(feature_names)} names for {X_selected_train.shape[1]} matrix columns"
    )

xgb_importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": xgb_model.feature_importances_,
})
xgb_importance_df = xgb_importance_df.loc[xgb_importance_df["importance"] > 0].sort_values(
    "importance", ascending=False
)


def feature_group(feature):
    if feature.startswith("genre::"):
        return "genre"
    if feature.startswith("tag::"):
        return "tag"
    if feature.startswith("category::"):
        return "category"
    if feature in BOOLEAN_FEATURES:
        return "platform"
    return "numeric_metadata"


xgb_importance_df["group"] = xgb_importance_df["feature"].apply(feature_group)
xgb_group_importance_df = (
    xgb_importance_df.groupby("group", as_index=False)
    .agg(total_importance=("importance", "sum"), top_feature=("feature", "first"))
    .sort_values("total_importance", ascending=False)
)

print("Top individual XGBoost features")
display(xgb_importance_df.head(25).round(4))
print("Importance by feature group")
display(xgb_group_importance_df.round(4))


## 12. Player-facing post-release value model

The developer-facing market model uses only features that could be known before launch. A player-facing value model can use post-release evidence because players see reviews, recommendations, concurrent-player activity, playtime, and critic signals when deciding whether the current list price feels justified.

This model keeps the same selected market scope and target, then adds post-release outcome features on top of the pre-release metadata. The interpretation changes: the prediction is an outcome-adjusted value price, not a launch-pricing recommendation. Ridge stays as the transparent baseline and XGBoost becomes the stronger post-release benchmark.

In [ ]:
POST_RELEASE_FEATURES = [
    "positive_share",
    "log1p_review_count",
    "log1p_recommendations",
    "log1p_peak_ccu",
    "log1p_avg_playtime_forever",
    "log1p_median_playtime_forever",
    "Metacritic score",
]

post_release_feature_summary = selected_train_df[POST_RELEASE_FEATURES].agg(["count", "median", "mean"]).T
post_release_feature_summary["missing_pct"] = (
    selected_train_df[POST_RELEASE_FEATURES].isna().mean() * 100
)
post_release_feature_summary["nonzero_pct"] = (
    selected_train_df[POST_RELEASE_FEATURES].fillna(0).ne(0).mean() * 100
)
post_release_feature_summary[["count", "missing_pct", "nonzero_pct", "median", "mean"]].round(3)

In [ ]:
post_release_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

X_post_release_train = sparse.csr_matrix(
    post_release_transformer.fit_transform(selected_train_df[POST_RELEASE_FEATURES])
)
X_post_release_test = sparse.csr_matrix(
    post_release_transformer.transform(selected_test_df[POST_RELEASE_FEATURES])
)

X_player_train = sparse.hstack([X_selected_train, X_post_release_train], format="csr")
X_player_test = sparse.hstack([X_selected_test, X_post_release_test], format="csr")

y_selected_train = selected_train_df[TARGET].to_numpy()

player_ridge_model = Ridge(alpha=10.0, random_state=42)
player_ridge_model.fit(X_player_train, y_selected_train)
player_ridge_pred_log = player_ridge_model.predict(X_player_test)

player_metrics = evaluate_predictions(
    f"Player-facing Ridge + post-release ({SELECTED_SCOPE})",
    y_selected_test,
    player_ridge_pred_log,
)
player_metrics["mae_improvement_vs_scope_median"] = selected_baseline_mae - player_metrics["mae_dollars"]
player_metrics["mae_improvement_pct_vs_scope_median"] = (
    player_metrics["mae_improvement_vs_scope_median"] / selected_baseline_mae * 100
)

player_results_df = pd.concat([
    selected_results_df,
    pd.DataFrame([player_metrics]),
], ignore_index=True)
player_results_df[[
    "model", "mae_dollars", "median_ae_dollars", "rmse_dollars",
    "pct_within_25pct", "pct_within_50pct",
    "mae_improvement_vs_scope_median", "mae_improvement_pct_vs_scope_median",
]].round(3)

In [ ]:
try:
    import xgboost as xgb
except ImportError as exc:
    raise ImportError(
        "xgboost is not installed in the active environment. Install it from requirements.txt and rerun this cell."
    ) from exc

player_xgb_model = xgb.XGBRegressor(
    objective="reg:squarederror",
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=1.0,
    tree_method="hist",
    random_state=42,
    n_jobs=-1,
)
player_xgb_model.fit(X_player_train, y_selected_train)
player_xgb_pred_log = player_xgb_model.predict(X_player_test)

player_xgb_metrics = evaluate_predictions(
    f"Player-facing XGBoost + post-release ({SELECTED_SCOPE})",
    y_selected_test,
    player_xgb_pred_log,
)
player_xgb_metrics["mae_improvement_vs_scope_median"] = selected_baseline_mae - player_xgb_metrics["mae_dollars"]
player_xgb_metrics["mae_improvement_pct_vs_scope_median"] = (
    player_xgb_metrics["mae_improvement_vs_scope_median"] / selected_baseline_mae * 100
)

player_post_release_results_df = pd.DataFrame([
    player_metrics,
    player_xgb_metrics,
])[[
    "model", "mae_dollars", "median_ae_dollars", "rmse_dollars",
    "pct_within_25pct", "pct_within_50pct",
    "mae_improvement_vs_scope_median", "mae_improvement_pct_vs_scope_median",
]]
player_post_release_results_df.round(3)

In [ ]:
player_model_comparison_df = player_post_release_results_df.copy()
BEST_PLAYER_MODEL_NAME = player_model_comparison_df.sort_values("mae_dollars").iloc[0]["model"]
best_player_pred_log = player_xgb_pred_log if BEST_PLAYER_MODEL_NAME.startswith("Player-facing XGBoost") else player_ridge_pred_log

print("Best post-release model:", BEST_PLAYER_MODEL_NAME)
player_model_comparison_df.round(3)

In [ ]:
PLAYER_VALUE_LOWER = 0.6
PLAYER_VALUE_UPPER = 1.5

player_value_df = selected_test_df[[
    "Name", "estimated_list_price", "Discount", "price_target_source", "review_count",
    "positive_share", "log1p_recommendations", "log1p_peak_ccu",
    "log1p_avg_playtime_forever", "Metacritic score", "Genres", "Tags",
]].copy()
player_value_df["predicted_player_value_price"] = to_price(best_player_pred_log)
player_value_df["player_value_ratio"] = (
    player_value_df["estimated_list_price"] / player_value_df["predicted_player_value_price"]
)
player_value_df["player_value_label"] = pd.cut(
    player_value_df["player_value_ratio"],
    bins=[0, PLAYER_VALUE_LOWER, PLAYER_VALUE_UPPER, np.inf],
    labels=["good-value", "fair-value", "weak-value"],
)
player_value_df["player_value_model"] = BEST_PLAYER_MODEL_NAME

player_value_summary = (
    player_value_df.groupby("player_value_label", observed=False)
    .agg(
        games=("Name", "size"),
        median_actual_price=("estimated_list_price", "median"),
        median_value_price=("predicted_player_value_price", "median"),
        median_value_ratio=("player_value_ratio", "median"),
        median_reviews=("review_count", "median"),
        median_positive_share=("positive_share", "median"),
    )
)

player_value_summary.round(3)

In [ ]:
player_value_df.sort_values(
    ["log1p_recommendations", "review_count"], ascending=False
).loc[:, [
    "Name", "estimated_list_price", "predicted_player_value_price", "player_value_ratio",
    "player_value_label", "Discount", "review_count", "positive_share",
    "log1p_recommendations", "log1p_peak_ccu", "Metacritic score",
]].head(20).round(3)

## 13. Popular-title outlier audit

This audit is for the questions that come up around high-profile titles. It looks at the 20 most-reviewed games in the selected test split, then orders them by the largest gap between actual and predicted price so we can explain why a big title is being flagged as over- or underpriced.

In [ ]:
POPULAR_OUTLIER_TOP_N = 20
POPULAR_OUTLIER_LOWER = 0.60
POPULAR_OUTLIER_UPPER = 1.50


def build_popular_outlier_audit(frame, predicted_log, predicted_price_col, label_col):
    audit = frame[[
        "Name", "estimated_list_price", "Discount", "price_target_source", "review_count",
        "log1p_recommendations", "positive_share", "log1p_peak_ccu",
        "log1p_avg_playtime_forever", "Metacritic score", "Genres", "Tags",
    ]].copy()
    audit[predicted_price_col] = to_price(predicted_log)
    audit["price_ratio"] = audit["estimated_list_price"] / audit[predicted_price_col]
    audit[label_col] = pd.cut(
        audit["price_ratio"],
        bins=[0, POPULAR_OUTLIER_LOWER, POPULAR_OUTLIER_UPPER, np.inf],
        labels=["below-market", "market-aligned", "above-market"],
    )
    audit["abs_price_error"] = (audit["estimated_list_price"] - audit[predicted_price_col]).abs()
    audit["abs_log_price_gap"] = np.abs(np.log(audit["price_ratio"]))

    popular = audit.nlargest(POPULAR_OUTLIER_TOP_N, "review_count").sort_values(
        ["abs_log_price_gap", "review_count"], ascending=[False, False]
    )
    outliers = popular.loc[popular[label_col].ne("market-aligned")].copy()
    return popular, outliers


POPULAR_DEV_AUDIT_COLS = [
    "Name", "estimated_list_price", "predicted_market_price", "price_ratio",
    "price_alignment", "abs_price_error", "Discount", "price_target_source",
    "review_count", "log1p_recommendations", "Metacritic score",
]

POPULAR_PLAYER_AUDIT_COLS = [
    "Name", "estimated_list_price", "predicted_player_value_price", "price_ratio",
    "player_value_label", "abs_price_error", "Discount", "price_target_source",
    "review_count", "log1p_recommendations", "Metacritic score",
]

dev_popular_audit_df, dev_popular_outliers_df = build_popular_outlier_audit(
    selected_test_df,
    selected_knn_pred_log,
    "predicted_market_price",
    "price_alignment",
)
player_popular_audit_df, player_popular_outliers_df = build_popular_outlier_audit(
    selected_test_df,
    best_player_pred_log,
    "predicted_player_value_price",
    "player_value_label",
)

print("Dev model: 20 most-reviewed titles in the test split, ordered by largest price gap")
display(dev_popular_audit_df.loc[:, POPULAR_DEV_AUDIT_COLS].round(3))
print("Dev model: popular titles outside the alignment band")
display(dev_popular_outliers_df.loc[:, POPULAR_DEV_AUDIT_COLS].round(3))

print("Player model: 20 most-reviewed titles in the test split, ordered by largest price gap")
display(player_popular_audit_df.loc[:, POPULAR_PLAYER_AUDIT_COLS].round(3))
print("Player model: popular titles outside the alignment band")
display(player_popular_outliers_df.loc[:, POPULAR_PLAYER_AUDIT_COLS].round(3))


## 14. Final takeaways

The notebook now points to two clear working setups and a defensible presentation story:

- Problem: estimate a market-aligned paid-game list price for Steam games, then label actual pricing as below-market, market-aligned, or above-market.
- Metric: use dollar MAE as the main metric because it directly measures expected pricing miss; use median AE, RMSE, and within-threshold rates as supporting diagnostics.
- Developer-facing launch/benchmark pricing: use `review_count >= 10` as the main market scope, keep pre-release metadata only, and report Ridge/KNN as explainable baselines with XGBoost as the strongest predictive benchmark.
- Player-facing value assessment: use the same selected scope, but add post-release review, recommendation, CCU, playtime, and Metacritic features because those signals are visible after launch. XGBoost is the current post-release winner, with Ridge kept as the simpler baseline.
- Feature interpretation: use the XGBoost feature-importance table to explain which metadata groups drive the boosted model, while avoiding causal claims.
- Use `0.6-1.5` as the fixed business-facing price-alignment rule. `0.75-1.25` was too strict, while `0.6-1.5` kept the label balance closer to a practical marketplace view.
- Keep percentile labels as a diagnostic only. They are useful for residual inspection, but they are not the main product output.
- Interpret the post-release model separately: it should not be used to recommend a launch price, but it can support player-facing strong/fair/weak value labels.
- Use the best post-release model for the value labels; in the current run, that is XGBoost.
- Use the popular-title outlier audit when a blockbuster looks mislabeled. It shows the model's predicted price, the ratio, and whether the title sits outside the alignment band.

The practical story for the final write-up is now coherent: start with simple explainable pre-release baselines, add a stronger boosted benchmark for accuracy, then add a separate post-release player-facing value layer.